# 10. Zwischenergebnisse merken

In [Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/02_backtracking.ipynb) hast du gesehen, wie eine rekursive Funktion alle Möglichkeiten
durchgeht. Oft ruft sie dabei **dieselbe Teilaufgabe immer wieder** auf, und die Aufrufe werden unvorstellbar viele. In
learn_jwinf3, Notebook 22, hast du die Abhilfe schon bei den Fibonacci-Zahlen kennengelernt: ein Wörterbuch, in dem man
Ergebnisse **merkt**. Dieses Notebook macht daraus ein Werkzeug für Wettbewerbsaufgaben (auf Englisch **Memoisierung**,
ein Kernstück der **dynamischen Programmierung**).

In diesem Notebook lernst du:

- eine rekursive Funktion so zu ergänzen, dass jedes Teilproblem nur **einmal** gerechnet wird,
- **Tupel als Schlüssel** zu benutzen, wenn ein Teilproblem durch mehrere Zahlen beschrieben wird,
- drei Arten von Antworten zu berechnen: **zählen** (Summe), **das Beste** (Minimum) und **geht es überhaupt?** (`or`),
- die **Aufrufe zu zählen** und die Laufzeit abzuschätzen: *Zahl der Teilprobleme × Arbeit je Teilproblem*,
- die **beste Lösung selbst** (nicht nur ihren Wert) zurückzuverfolgen,
- das Wörterbuch mit `@lru_cache` automatisch führen zu lassen, und die Fallen dabei,
- zwei typische Fehler zu erkennen.

In der Wettbewerbs-Ecke berechnest du die billigsten Eintrittskarten für eine Gruppe, und für Schnelle teilst du Nummern in
kurze Blöcke.

## 1. Treppenstufen: Wege zählen

Eine Treppe hat `n` Stufen. Du darfst in jedem Schritt **1, 2 oder 3 Stufen** nach oben gehen. Auf wie viele verschiedene
Arten kommst du oben an? Für `n = 3` sind es 4 Arten: 1+1+1, 1+2, 2+1 und 3.

Die letzte Stufe, die du betrittst, erreichst du mit einem Schritt von 1, 2 oder 3 Stufen. Also gilt
`wege(n) = wege(n - 1) + wege(n - 2) + wege(n - 3)`. Dazu kommen zwei einfache Fälle: Bei `n = 0` bist du schon oben (genau
**eine** Art: nichts mehr tun), bei `n < 0` bist du zu weit gesprungen (**null** Arten).

Die Funktion `treppe_langsam` ist fertig. Der Zähler `aufrufe` ist eine Liste mit einem Element, damit die Funktion ihn
verändern darf (wie in Notebook 02). Ergänze `treppe_schnell` so, dass sie das Ergebnis für jedes `n` im Wörterbuch `gemerkt`
**aufbewahrt**: Erst nachsehen, ob `n` schon im Wörterbuch steht, sonst rechnen und eintragen.

#### Aufgabe 1.1

Schreibe `treppe_schnell(n)` mit dem Wörterbuch `gemerkt`. Zähle in `treppe_schnell` ebenfalls die Aufrufe.

**Erwartete Ausgabe:**
```
n = 5 : 13 Wege, 46 Aufrufe ohne Merken, 16 mit Merken
n = 10 : 274 Wege, 979 Aufrufe ohne Merken, 31 mit Merken
n = 15 : 5768 Wege, 20617 Aufrufe ohne Merken, 46 mit Merken
```

In [ ]:
aufrufe = [0]

def treppe_langsam(n):
    aufrufe[0] = aufrufe[0] + 1
    if n < 0:
        return 0
    if n == 0:
        return 1
    return treppe_langsam(n - 1) + treppe_langsam(n - 2) + treppe_langsam(n - 3)

gemerkt = {}

def treppe_schnell(n):
    aufrufe[0] = aufrufe[0] + 1
    return 0                             # ersetze diese Zeile

for n in [5, 10, 15]:
    aufrufe[0] = 0
    wege = treppe_langsam(n)
    langsam = aufrufe[0]
    gemerkt.clear()                      # Wörterbuch für jede Treppe leeren
    aufrufe[0] = 0
    treppe_schnell(n)
    print('n =', n, ':', wege, 'Wege,', langsam, 'Aufrufe ohne Merken,', aufrufe[0], 'mit Merken')

<details>
<summary>Tipp anzeigen</summary>

Erst die einfachen Fälle (`n < 0`, `n == 0`), dann `if n in gemerkt: return gemerkt[n]`. Danach das Ergebnis in einer
Variablen berechnen, mit `gemerkt[n] = ergebnis` eintragen und zurückgeben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
aufrufe = [0]

def treppe_langsam(n):
    aufrufe[0] = aufrufe[0] + 1
    if n < 0:
        return 0
    if n == 0:
        return 1
    return treppe_langsam(n - 1) + treppe_langsam(n - 2) + treppe_langsam(n - 3)

gemerkt = {}                             # n -> Zahl der Wege

def treppe_schnell(n):
    aufrufe[0] = aufrufe[0] + 1
    if n < 0:
        return 0
    if n == 0:
        return 1
    if n in gemerkt:                     # schon berechnet: sofort zurückgeben
        return gemerkt[n]
    ergebnis = treppe_schnell(n - 1) + treppe_schnell(n - 2) + treppe_schnell(n - 3)
    gemerkt[n] = ergebnis                # eintragen, bevor man zurückkehrt
    return ergebnis

for n in [5, 10, 15]:
    aufrufe[0] = 0
    wege = treppe_langsam(n)
    langsam = aufrufe[0]
    gemerkt.clear()                      # Wörterbuch für jede Treppe leeren
    aufrufe[0] = 0
    treppe_schnell(n)
    print('n =', n, ':', wege, 'Wege,', langsam, 'Aufrufe ohne Merken,', aufrufe[0], 'mit Merken')
```

Die Ergebnisse sind gleich, aber die Aufrufe unterscheiden sich gewaltig: Bei `n = 15` sind es rund 20 000 ohne und nur 46
mit Merken. Das Muster ist immer dasselbe: **Nachsehen, rechnen, eintragen.** Das Ergebnis darf erst eingetragen werden, wenn
es fertig berechnet ist.

</details>

Mit dem Wörterbuch geht auch eine riesige Treppe. Wir schreiben `treppe_schnell` noch einmal ohne Zähler und rufen sie auf:

In [ ]:
gemerkt = {}

def treppe_schnell(n):
    if n < 0:
        return 0
    if n == 0:
        return 1
    if n not in gemerkt:
        gemerkt[n] = treppe_schnell(n - 1) + treppe_schnell(n - 2) + treppe_schnell(n - 3)
    return gemerkt[n]

print(treppe_schnell(100))

#### Frage 1.2

1. Bei `n = 5` hat `treppe_schnell` 16 Aufrufe, bei `n = 10` 31, bei `n = 15` 46. Wie viele wären es bei `n = 100`? Wie viele
   **verschiedene Teilprobleme** (verschiedene Werte von `n`) gibt es, und wie viel Arbeit macht jedes?
2. Ohne Merken wuchsen die Aufrufe von 979 (`n = 10`) auf 20 617 (`n = 15`). Ungefähr um welchen Faktor wächst die Zahl der
   Aufrufe mit jeder zusätzlichen Stufe? Schätze damit die Aufrufe für `n = 40`.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Aufrufe wachsen um 3 je Stufe: 16, 31, 46, also **3n + 1**. Für `n = 100` sind es **301**. Es gibt nur die
   Teilprobleme `n, n - 1, ..., 1` (und den Rand `0`, `-1`, `-2`). Jedes wird **einmal** berechnet, und die Arbeit besteht
   aus **drei** Aufrufen und zwei Additionen. Allgemein gilt die Faustregel: **Laufzeit = Zahl der verschiedenen
   Teilprobleme × Arbeit je Teilproblem**, hier n · 3, also **O(n)**.
2. 20 617 / 979 ≈ 21 für fünf Stufen, das sind ungefähr 1,84 je Stufe (denn 1,84⁵ ≈ 21). Die Zahl **verdoppelt sich fast mit
   jeder Stufe**: Das ist ein **exponentielles** Wachstum. Für 30 weitere Stufen ergibt das etwa 979 · 21⁶ ≈ 85 Milliarden
   Aufrufe für `n = 40`, viel zu viel. Das Wörterbuch macht aus einem exponentiellen ein **lineares** Verfahren.

</details>

## 2. Wege im Gitter: Tupel als Schlüssel

In einem Gitter mit `z + 1` Zeilen und `s + 1` Spalten (Nummerierung ab 0) gehst du vom Feld `(0, 0)` zum Feld `(z, s)`. Du
darfst in jedem Schritt **nach unten** (Zeile + 1) oder **nach rechts** (Spalte + 1) gehen. Einige Felder sind **Hindernisse**,
sie darf man nicht betreten.

Rückwärts gedacht: Auf das Feld `(z, s)` kommt man von oben `(z - 1, s)` oder von links `(z, s - 1)`. Also ist
`wege(z, s) = wege(z - 1, s) + wege(z, s - 1)`. Ein Hindernis hat 0 Wege, das Startfeld genau 1.

Jetzt wird das Teilproblem durch **zwei** Zahlen beschrieben. Als Schlüssel im Wörterbuch nimmst du ein **Tupel**
`(z, s)`: Tupel sind unveränderlich und deshalb als Schlüssel erlaubt (Listen nicht, mehr dazu in Abschnitt 6). Die
Hindernisse speichern wir als Menge von Tupeln, `(2, 3) in HINDERNISSE` ist die Abfrage.

#### Aufgabe 2.1

Schreibe `gitter_schnell(z, s)` mit dem Wörterbuch `gemerkt_gitter` und dem Tupel `(z, s)` als Schlüssel. `gitter_langsam` ist
fertig.

**Erwartete Ausgabe:**
```
8 x 8 , 0 Hindernisse: 12870 Wege, 48619 Aufrufe ohne Merken, 145 mit Merken
5 x 5 , 3 Hindernisse: 51 Wege, 370 Aufrufe ohne Merken, 56 mit Merken
```

In [ ]:
aufrufe = [0]
HINDERNISSE = set()

def gitter_langsam(z, s):
    aufrufe[0] = aufrufe[0] + 1
    if (z, s) in HINDERNISSE:
        return 0
    if z == 0 and s == 0:
        return 1
    summe = 0
    if z > 0:
        summe = summe + gitter_langsam(z - 1, s)
    if s > 0:
        summe = summe + gitter_langsam(z, s - 1)
    return summe

gemerkt_gitter = {}

def gitter_schnell(z, s):
    aufrufe[0] = aufrufe[0] + 1
    return 0                             # ersetze diese Zeile

def zeige_gitter(z, s):
    aufrufe[0] = 0
    wege = gitter_langsam(z, s)
    langsam = aufrufe[0]
    gemerkt_gitter.clear()
    aufrufe[0] = 0
    wege_schnell = gitter_schnell(z, s)
    print(z, 'x', s, ',', len(HINDERNISSE), 'Hindernisse:', wege, 'Wege,', langsam, 'Aufrufe ohne Merken,', aufrufe[0], 'mit Merken')

zeige_gitter(8, 8)
HINDERNISSE = {(1, 1), (2, 3), (3, 0)}
zeige_gitter(5, 5)

<details>
<summary>Tipp anzeigen</summary>

Wie in 1.1: erst die einfachen Fälle (Hindernis, Start), dann `schluessel = (z, s)` bilden und im Wörterbuch nachsehen. Zum Schluss
`gemerkt_gitter[schluessel] = summe`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
aufrufe = [0]
HINDERNISSE = set()

def gitter_langsam(z, s):
    aufrufe[0] = aufrufe[0] + 1
    if (z, s) in HINDERNISSE:
        return 0
    if z == 0 and s == 0:
        return 1
    summe = 0
    if z > 0:
        summe = summe + gitter_langsam(z - 1, s)
    if s > 0:
        summe = summe + gitter_langsam(z, s - 1)
    return summe

gemerkt_gitter = {}                      # (z, s) -> Zahl der Wege

def gitter_schnell(z, s):
    aufrufe[0] = aufrufe[0] + 1
    if (z, s) in HINDERNISSE:
        return 0
    if z == 0 and s == 0:
        return 1
    schluessel = (z, s)                  # Tupel als Schlüssel
    if schluessel in gemerkt_gitter:
        return gemerkt_gitter[schluessel]
    summe = 0
    if z > 0:
        summe = summe + gitter_schnell(z - 1, s)
    if s > 0:
        summe = summe + gitter_schnell(z, s - 1)
    gemerkt_gitter[schluessel] = summe
    return summe

def zeige_gitter(z, s):
    aufrufe[0] = 0
    wege = gitter_langsam(z, s)
    langsam = aufrufe[0]
    gemerkt_gitter.clear()
    aufrufe[0] = 0
    wege_schnell = gitter_schnell(z, s)
    print(z, 'x', s, ',', len(HINDERNISSE), 'Hindernisse:', wege, 'Wege,', langsam, 'Aufrufe ohne Merken,', aufrufe[0], 'mit Merken')

zeige_gitter(8, 8)
HINDERNISSE = {(1, 1), (2, 3), (3, 0)}
zeige_gitter(5, 5)
```

Das Ergebnis `12870` für das 8×8-Gitter ohne Hindernisse ist auch die Zahl der Möglichkeiten, aus 16 Schritten 8 „nach unten“
auszuwählen (`math.comb(16, 8)` aus Notebook 01). Das Wörterbuch macht aus fast 50 000 Aufrufen nur 145.

</details>

#### Frage 2.2

1. Wie viele verschiedene Teilprobleme `(z, s)` gibt es im 8×8-Gitter höchstens, und wie viel Arbeit macht jedes? Passt das zu
   den 145 Aufrufen?
2. Wie viele Teilprobleme hat ein 100×100-Gitter? Geht es noch mit Merken? Und ein Gitter mit 600 mal 600 Feldern?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Zeilen 0 bis 8 und Spalten 0 bis 8: höchstens **9 · 9 = 81** Teilprobleme. Jedes macht höchstens zwei Aufrufe (von oben und
   von links). Zusammen sind das etwa 2 · 81 = 162 Aufrufe, tatsächlich sind es 145, weil es am Rand nur einen Nachbarn gibt.
   Laufzeit: **Teilprobleme × Arbeit = O(z · s)**.
2. 101 · 101 = **10 201** Teilprobleme: mit Merken kein Problem. Die Rekursion geht beim Aufruf von `(100, 100)` über `(99, 100)`
   bis `(0, 0)` nur **200 Ebenen tief**, das erlaubt Python (etwa 1000 sind möglich). Bei 600 mal 600 Feldern wären es 1200 Ebenen,
   dann stürzt das Programm mit einem `RecursionError` ab, obwohl der Speicher für 360 000 Teilprobleme reichen würde. Für solche
   Fälle füllt man die Tabelle mit Schleifen von klein nach groß (Notebook 11).

</details>

## 3. Das Beste und „geht es überhaupt?“

Bisher hast du **gezählt**: Die Antwort ist die **Summe** der Antworten der Teilprobleme. Das Wörterbuch hilft bei jeder
Art von Antwort:

| Frage | So kombiniert man die Teilantworten |
|---|---|
| Wie viele Möglichkeiten gibt es? | Summe (`+`) |
| Was ist das **Beste**: wenigste, billigste, kürzeste? | Minimum (`min`) oder Maximum (`max`) |
| **Geht es überhaupt?** | `or` (geht in einem der Fälle?) |

Beispiel **Münzen**: In einem Land gibt es nur die Münzen `MUENZEN = (1, 4, 6)`. Wie viele Münzen braucht man **mindestens**,
um einen Betrag genau zu zahlen? Probiere jede Münze als **erste** aus: `wenigste(betrag)` ist 1 plus das Beste von
`wenigste(betrag - münze)` über alle Münzen, die nicht größer als `betrag` sind. Für den Betrag 0 braucht man 0 Münzen. Mit anderen Münzen kann es sein, dass ein Betrag **gar nicht**
zu zahlen ist (etwa 11 mit den Münzen 5 und 8). Dann soll die Funktion `None` zurückgeben. Weil `None` auch ein gültiger Wert im
Wörterbuch ist, schaust du mit `in` nach, nicht mit `if gemerkt_muenze[betrag]`.

#### Aufgabe 3.1

Schreibe `muenze_schnell(betrag)` mit `gemerkt_muenze` (Schlüssel: der Betrag). Merke auch das Ergebnis `None`.

**Erwartete Ausgabe:**
```
Betrag 8 : 2 Münzen, 31 Aufrufe ohne Merken, 17 mit Merken
Betrag 12 : 2 Münzen, 143 Aufrufe ohne Merken, 29 mit Merken
Betrag 20 : 4 Münzen, 3053 Aufrufe ohne Merken, 53 mit Merken
Betrag 25 : 5 Münzen, 20643 Aufrufe ohne Merken, 68 mit Merken
mit 5 und 8: Betrag 11 -> None
mit 5 und 8: Betrag 18 -> 3
mit 5 und 8: Betrag 20 -> 4
```

In [ ]:
aufrufe = [0]
MUENZEN = (1, 4, 6)

def muenze_langsam(betrag):
    aufrufe[0] = aufrufe[0] + 1
    if betrag == 0:
        return 0
    beste = None
    for muenze in MUENZEN:
        if muenze <= betrag:
            rest = muenze_langsam(betrag - muenze)
            if rest is not None:                 # None heißt: Rest geht nicht
                if beste is None or rest + 1 < beste:
                    beste = rest + 1
    return beste

gemerkt_muenze = {}

def muenze_schnell(betrag):
    aufrufe[0] = aufrufe[0] + 1
    return 0                             # ersetze diese Zeile

for betrag in [8, 12, 20, 25]:
    aufrufe[0] = 0
    anzahl = muenze_langsam(betrag)
    langsam = aufrufe[0]
    gemerkt_muenze.clear()
    aufrufe[0] = 0
    muenze_schnell(betrag)
    print('Betrag', betrag, ':', anzahl, 'Münzen,', langsam, 'Aufrufe ohne Merken,', aufrufe[0], 'mit Merken')

MUENZEN = (5, 8)
gemerkt_muenze.clear()
for betrag in [11, 18, 20]:
    print('mit 5 und 8: Betrag', betrag, '->', muenze_schnell(betrag))

<details>
<summary>Tipp anzeigen</summary>

Wie in 1.1: nach dem Fall `betrag == 0` mit `if betrag in gemerkt_muenze` nachsehen. Am Ende `gemerkt_muenze[betrag] = beste`
eintragen, **auch wenn `beste` noch `None` ist**.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
aufrufe = [0]
MUENZEN = (1, 4, 6)

def muenze_langsam(betrag):
    aufrufe[0] = aufrufe[0] + 1
    if betrag == 0:
        return 0
    beste = None
    for muenze in MUENZEN:
        if muenze <= betrag:
            rest = muenze_langsam(betrag - muenze)
            if rest is not None:                 # None heißt: Rest geht nicht
                if beste is None or rest + 1 < beste:
                    beste = rest + 1
    return beste

gemerkt_muenze = {}                      # betrag -> wenigste Münzen (oder None)

def muenze_schnell(betrag):
    aufrufe[0] = aufrufe[0] + 1
    if betrag == 0:
        return 0
    if betrag in gemerkt_muenze:         # auch ein gemerktes None zählt
        return gemerkt_muenze[betrag]
    beste = None
    for muenze in MUENZEN:
        if muenze <= betrag:
            rest = muenze_schnell(betrag - muenze)
            if rest is not None:
                if beste is None or rest + 1 < beste:
                    beste = rest + 1
    gemerkt_muenze[betrag] = beste
    return beste

for betrag in [8, 12, 20, 25]:
    aufrufe[0] = 0
    anzahl = muenze_langsam(betrag)
    langsam = aufrufe[0]
    gemerkt_muenze.clear()
    aufrufe[0] = 0
    muenze_schnell(betrag)
    print('Betrag', betrag, ':', anzahl, 'Münzen,', langsam, 'Aufrufe ohne Merken,', aufrufe[0], 'mit Merken')

MUENZEN = (5, 8)
gemerkt_muenze.clear()
for betrag in [11, 18, 20]:
    print('mit 5 und 8: Betrag', betrag, '->', muenze_schnell(betrag))
```

Für den Betrag 12 sind zwei Münzen von 6 am besten, für 20 vier Münzen (6 + 6 + 4 + 4) und für 25 fünf (6 + 6 + 6 + 6 + 1).
Ein gieriges Verfahren („immer die größte Münze“) hätte bei 8 eine 6 und zwei 1 genommen, also 3 Münzen statt 2: Hier muss man
wirklich alle Möglichkeiten für die erste Münze probieren. Bei den Münzen 5 und 8 sind 11 nicht, aber 18 (5 + 5 + 8) und 20
(4 · 5) zu zahlen. Die Zahl der Teilprobleme ist höchstens `betrag + 1`, die Arbeit je Teilproblem ist die Zahl der Münzsorten
(hier 3): O(betrag · Zahl der Sorten).

</details>

Jetzt die dritte Art, **„geht es überhaupt?“**: Gegeben sind Zahlen, jede darf **höchstens einmal** benutzt werden. Kann man
mit einigen von ihnen genau die Summe `rest` bilden? Das Teilproblem hat jetzt **zwei** Angaben: Welche Zahlen darf ich noch
benutzen (alle ab Position `i`) und wie viel Summe fehlt noch (`rest`)? Beide gehören in den Schlüssel:
`geht(i, rest)` ist `True`, wenn man mit den Zahlen ab Position `i` genau die Summe `rest` bilden kann.

Zwei Möglichkeiten für die Zahl an Position `i`: **benutzen** (`rest - zahlen[i]`) **oder nicht benutzen**. Es geht, wenn
**mindestens eine** der beiden Möglichkeiten geht, also `or`. Ist `rest == 0`, ist man fertig (`True`). Ist `rest` negativ oder
gibt es keine Zahlen mehr, geht es nicht (`False`).

#### Aufgabe 3.2

Schreibe `geht(i, rest)` mit `gemerkt_summe` und dem Tupel `(i, rest)` als Schlüssel. `geht_langsam` (ohne Merken) ist fertig.
Die Zahlen stehen in der Variablen `ZAHLEN`. Ein Zähler `aufrufe` gibt es wie vorher.

**Erwartete Ausgabe:**
```
Ziel 9 : True , 10 Aufrufe
Ziel 30 : False , 65 Aufrufe
Ziel 7 : True , 5 Aufrufe
Ziel 26 : True , 8 Aufrufe
14 gerade Zahlen, Ziel 101: False , 759 Aufrufe mit Merken
ohne Merken: 24919 Aufrufe
```

In [ ]:
def geht_langsam(i, rest):
    aufrufe[0] = aufrufe[0] + 1
    if rest == 0:
        return True
    if rest < 0 or i == len(ZAHLEN):
        return False
    return geht_langsam(i + 1, rest - ZAHLEN[i]) or geht_langsam(i + 1, rest)

gemerkt_summe = {}

def geht(i, rest):
    aufrufe[0] = aufrufe[0] + 1
    return False                         # ersetze diese Zeile

ZAHLEN = (3, 34, 4, 12, 5, 2)
for ziel in [9, 30, 7, 26]:
    aufrufe[0] = 0
    gemerkt_summe.clear()
    print('Ziel', ziel, ':', geht(0, ziel), ',', aufrufe[0], 'Aufrufe')

ZAHLEN = (2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28)
aufrufe[0] = 0
gemerkt_summe.clear()
print('14 gerade Zahlen, Ziel 101:', geht(0, 101), ',', aufrufe[0], 'Aufrufe mit Merken')
gemerkt_summe.clear()
aufrufe[0] = 0
geht_langsam(0, 101)
print('ohne Merken:', aufrufe[0], 'Aufrufe')

<details>
<summary>Tipp anzeigen</summary>

Genau wie `geht_langsam`, nur mit einem Schlüssel `(i, rest)`: Nach den einfachen Fällen nachsehen, sonst das Ergebnis des
`or` in einer Variablen berechnen, eintragen und zurückgeben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def geht_langsam(i, rest):
    aufrufe[0] = aufrufe[0] + 1
    if rest == 0:
        return True
    if rest < 0 or i == len(ZAHLEN):
        return False
    return geht_langsam(i + 1, rest - ZAHLEN[i]) or geht_langsam(i + 1, rest)

gemerkt_summe = {}                       # (i, rest) -> True oder False

def geht(i, rest):
    aufrufe[0] = aufrufe[0] + 1
    if rest == 0:
        return True
    if rest < 0 or i == len(ZAHLEN):
        return False
    schluessel = (i, rest)
    if schluessel in gemerkt_summe:
        return gemerkt_summe[schluessel]
    ergebnis = geht(i + 1, rest - ZAHLEN[i]) or geht(i + 1, rest)
    gemerkt_summe[schluessel] = ergebnis
    return ergebnis

ZAHLEN = (3, 34, 4, 12, 5, 2)
for ziel in [9, 30, 7, 26]:
    aufrufe[0] = 0
    gemerkt_summe.clear()
    print('Ziel', ziel, ':', geht(0, ziel), ',', aufrufe[0], 'Aufrufe')

ZAHLEN = (2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28)
aufrufe[0] = 0
gemerkt_summe.clear()
print('14 gerade Zahlen, Ziel 101:', geht(0, 101), ',', aufrufe[0], 'Aufrufe mit Merken')
gemerkt_summe.clear()
aufrufe[0] = 0
geht_langsam(0, 101)
print('ohne Merken:', aufrufe[0], 'Aufrufe')
```

Die Antworten stimmen: 9 = 4 + 5, 7 = 5 + 2, 26 = 3 + 4 + 12 + 5 + 2, aber 30 geht nicht (34 ist zu groß, und die anderen
Zahlen ergeben zusammen nur 26). Das Python-`or` hört auf, sobald die erste Möglichkeit `True` liefert, deshalb sind
die Aufrufe bei einem Erfolg oft weniger. **Wenn es nicht geht**, müssen alle Möglichkeiten probiert werden. Dann ist
der Unterschied groß: Bei den 14 geraden Zahlen und dem ungeraden Ziel 101 (unmöglich!) braucht die Version ohne Merken
24 919 Aufrufe, die mit Merken nur 759. Mit Merken gibt es höchstens 15 · 102 Teilprobleme `(i, rest)`, das Ergebnis jedes
Teilproblems braucht nur zwei Aufrufe.

</details>

## 4. Die beste Lösung zurückverfolgen

Meistens will man nicht nur wissen, **dass** 2 Münzen reichen, sondern **welche**. Dafür merkt man sich zu jedem Teilproblem
zusätzlich, **welche Möglichkeit die beste war**, hier: welche Münze man zuerst nimmt. Das ist wie `vorgaenger` bei der
Breitensuche in Notebook 05. Danach folgt man diesen Einträgen von hinten nach vorn: Nimm die gemerkte Münze, ziehe sie ab und
sieh für den Rest wieder nach, bis der Betrag 0 ist.

Man kann den Plan auch **nachträglich** finden, ohne etwas zusätzlich zu speichern: Für den Betrag probiert man noch einmal jede
Münze und nimmt eine, bei der `wenigste(betrag - münze) + 1 == wenigste(betrag)` gilt. Dank des Wörterbuchs ist das schnell.
Ein zweites Wörterbuch ist aber oft einfacher zu schreiben.

#### Aufgabe 4.1

Erweitere die Münzfunktion zu `muenze_plan(betrag)`: Sie gibt wie vorher die wenigste Münzzahl zurück (der Betrag ist immer
zu zahlen) und trägt in das Wörterbuch `wahl` ein, welche Münze man bei diesem Betrag zuerst nimmt. Schreibe dann
`zahle(betrag)`: Sie gibt die Liste der Münzen zurück, die man zahlen muss (nach dem Aufruf von `muenze_plan(betrag)`).

**Erwartete Ausgabe:**
```
8 [4, 4]
12 [6, 6]
20 [4, 4, 6, 6]
25 [1, 6, 6, 6, 6]
```

In [ ]:
MUENZEN = (1, 4, 6)
gemerkt_wahl = {}                        # betrag -> wenigste Münzen
wahl = {}                                # betrag -> erste Münze der besten Möglichkeit

def muenze_plan(betrag):
    return 0                             # ersetze diese Zeile

def zahle(betrag):
    liste = []
    # dein Code
    return liste

for betrag in [8, 12, 20, 25]:
    muenze_plan(betrag)
    print(betrag, zahle(betrag))

<details>
<summary>Tipp anzeigen</summary>

Wenn eine Möglichkeit besser ist als `beste`, setze `beste = anzahl` und `wahl[betrag] = muenze`. In `zahle`: Solange
`betrag > 0`, hänge `wahl[betrag]` an die Liste an und ziehe sie vom Betrag ab. Mit den Münzen (1, 4, 6) ist jeder Betrag zu zahlen
(wegen der 1), also gibt es kein `None`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
MUENZEN = (1, 4, 6)
gemerkt_wahl = {}                        # betrag -> wenigste Münzen
wahl = {}                                # betrag -> erste Münze der besten Möglichkeit

def muenze_plan(betrag):
    if betrag == 0:
        return 0
    if betrag in gemerkt_wahl:
        return gemerkt_wahl[betrag]
    beste = None
    for muenze in MUENZEN:
        if muenze <= betrag:
            anzahl = muenze_plan(betrag - muenze) + 1
            if beste is None or anzahl < beste:
                beste = anzahl
                wahl[betrag] = muenze    # diese Münze ist bis jetzt die beste
    gemerkt_wahl[betrag] = beste
    return beste

def zahle(betrag):
    liste = []
    while betrag > 0:
        muenze = wahl[betrag]            # die beste erste Münze für diesen Betrag
        liste.append(muenze)
        betrag = betrag - muenze
    return liste

for betrag in [8, 12, 20, 25]:
    muenze_plan(betrag)
    print(betrag, zahle(betrag))
```

Der Plan für 8 ist `[4, 4]`, für 25 `[1, 6, 6, 6, 6]`. Bei mehreren gleich guten Möglichkeiten gewinnt die zuerst probierte, hier die kleinere Münze. Die Zahl der Einträge ist jeweils die Münzzahl aus 3.1. Merke: Man speichert
nur **eine** Entscheidung je Teilproblem, nicht den ganzen Plan. Der Plan entsteht erst beim Zurückverfolgen. Das spart
Speicher und ist die übliche Methode bei Wettbewerbsaufgaben, bei denen man die **Lösung** ausgeben muss.

</details>

## 5. Abkürzung: `@lru_cache`

Das Wörterbuch hast du jetzt in jeder Funktion nach demselben Muster geschrieben:

1. Schlüssel bilden (das Tupel aller Argumente),
2. „steht der Schlüssel schon im Wörterbuch? Dann das gemerkte Ergebnis zurückgeben“,
3. sonst rechnen, **eintragen**, zurückgeben.

Genau dieses Muster kann Python für dich übernehmen. In der Bibliothek `functools` gibt es einen **Dekorator** namens
`lru_cache`. Ein Dekorator ist eine Zeile **über** `def`, die einer Funktion eine Eigenschaft hinzufügt:

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def treppe(n):
    if n < 0:
        return 0
    if n == 0:
        return 1
    return treppe(n - 1) + treppe(n - 2) + treppe(n - 3)
```

Du schreibst die Funktion **ohne** Wörterbuch, so wie `treppe_langsam`. Der Dekorator ersetzt

- das **Wörterbuch** `gemerkt` (es liegt verborgen in der Funktion; der Schlüssel ist das Tupel der Argumente),
- die **Abfrage** „schon berechnet?“ am Anfang und
- das **Eintragen** am Ende.

`maxsize=None` heißt: nichts vergessen, der Speicher ist nicht begrenzt. (Ohne diese Angabe merkt sich Python nur die letzten
128 Ergebnisse. Das „LRU“ im Namen steht für „least recently used“: Das am längsten ungenutzte Ergebnis fliegt zuerst raus.
Seit Python 3.9 gibt es dafür auch die Kurzform `@cache`.)

Drei Dinge musst du wissen:

- Die **Argumente** werden als Schlüssel benutzt, sie müssen also **unveränderlich** sein: Zahlen, Texte, **Tupel**. Eine Liste als
  Argument ergibt einen Fehler (Abschnitt 6).
- Der Speicher **bleibt** zwischen den Aufrufen erhalten. Mit `treppe.cache_clear()` leerst du ihn, mit `treppe.cache_info()` siehst
  du, wie oft ein Ergebnis aus dem Speicher kam (`hits`) und wie oft gerechnet wurde (`misses`).
- Der Dekorator merkt sich nur das **Ergebnis zu den Argumenten**. Liest die Funktion eine **globale Variable**, die sich später
  ändert, kennt der Speicher die alte Antwort weiter. Das ist eine Falle (Aufgabe 5.2).

#### Aufgabe 5.1

Setze den Dekorator `@lru_cache(maxsize=None)` über `treppe_lru` (im Gerüst fehlt die Zeile) und gib danach
`treppe_lru.cache_info()` aus.

**Erwartete Ausgabe:**
```
5768 18 Aufrufe
CacheInfo(hits=28, misses=18, maxsize=None, currsize=18)
```

In [ ]:
from functools import lru_cache

aufrufe = [0]

# hier fehlt eine Zeile
def treppe_lru(n):
    aufrufe[0] = aufrufe[0] + 1
    if n < 0:
        return 0
    if n == 0:
        return 1
    return treppe_lru(n - 1) + treppe_lru(n - 2) + treppe_lru(n - 3)

print(treppe_lru(15), aufrufe[0], 'Aufrufe')

<details>
<summary>Tipp anzeigen</summary>

Die Zeile `@lru_cache(maxsize=None)` steht direkt über `def treppe_lru(n):`, ohne Leerzeile. Der Import steht schon oben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from functools import lru_cache

aufrufe = [0]

@lru_cache(maxsize=None)
def treppe_lru(n):
    aufrufe[0] = aufrufe[0] + 1          # wird nur ausgeführt, wenn gerechnet wird
    if n < 0:
        return 0
    if n == 0:
        return 1
    return treppe_lru(n - 1) + treppe_lru(n - 2) + treppe_lru(n - 3)

print(treppe_lru(15), aufrufe[0], 'Aufrufe')
print(treppe_lru.cache_info())
```

Ohne den Dekorator wären es 20 617 Aufrufe. Jetzt führt Python den Rumpf nur **18-mal** aus, einmal für jedes der 18 verschiedenen
Argumente 15, 14, ..., 0, -1, -2 (`misses=18`). Die übrigen 28 Aufrufe holt es aus dem Speicher (`hits=28`).
`hits + misses = 46` sind genau die 46 Aufrufe der Version mit eigenem Wörterbuch aus 1.1. Der Zähler `aufrufe` steht im
Rumpf und zählt deshalb nur die Fälle, in denen wirklich gerechnet wird.

</details>

#### Aufgabe 5.2: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
from functools import lru_cache

HINDERNISSE = set()

@lru_cache(maxsize=None)
def gitter(z, s):
    if (z, s) in HINDERNISSE:
        return 0
    if z == 0 and s == 0:
        return 1
    summe = 0
    if z > 0:
        summe = summe + gitter(z - 1, s)
    if s > 0:
        summe = summe + gitter(z, s - 1)
    return summe

print(gitter(2, 2))
HINDERNISSE = {(1, 1)}
print(gitter(2, 2))
gitter.cache_clear()
print(gitter(2, 2))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
6
6
2
```

Ausgegeben wird `6`, dann **wieder `6`**, dann `2`. Beim zweiten Aufruf hat Python die Antwort für `(2, 2)` im Speicher gefunden und
die Funktion gar nicht mehr ausgeführt, obwohl sich die Hindernisse inzwischen geändert haben. Der Dekorator kennt nur die
**Argumente**, nicht die globale Variable `HINDERNISSE`. Erst `cache_clear()` leert den Speicher, dann stimmt die
Antwort 2 (mit dem Hindernis in der Mitte bleiben nur die Wege über den Rand). Die Regel: **Alles, wovon das Ergebnis
abhängt, muss ein Argument sein**, oder der Speicher muss geleert werden, sobald sich etwas ändert. Bei eigenen Wörterbüchern
gilt dasselbe (Abschnitt 6). Bei mehreren Wettbewerbs-Beispielen nacheinander gehört deshalb zwischen die Beispiele ein
`cache_clear()` beziehungsweise `gemerkt.clear()`.

</details>

## 6. Finde den Fehler

Zwei Fehler passieren beim Merken besonders oft.

**Fehler 1: eine Liste als Schlüssel.** Beide Programme enden mit derselben Fehlermeldung:

```python
gemerkt = {}
position = [2, 3]
gemerkt[position] = 5
```

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def summe(zahlen):
    return sum(zahlen)

print(summe([1, 2, 3]))
```

Fehlermeldung: `TypeError: unhashable type: 'list'`

Eine Liste kann man verändern, deshalb darf sie kein Schlüssel sein (Python könnte den Platz im Wörterbuch nicht mehr finden, wenn
sich die Liste später ändert). Abhilfe: ein **Tupel** verwenden, `gemerkt[(2, 3)] = 5` beziehungsweise `summe((1, 2, 3))`. Hast du
eine Liste `liste`, macht `tuple(liste)` ein Tupel daraus.

**Fehler 2: ein Schlüssel, der nicht alles enthält.**

#### Aufgabe 6.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
ZAHLEN = (1, 3, 2, 2)
gemerkt = {}

def geht_falsch(i, rest):
    if rest == 0:
        return True
    if rest < 0 or i == len(ZAHLEN):
        return False
    if rest in gemerkt:                  # Schlüssel ist nur rest
        return gemerkt[rest]
    ergebnis = geht_falsch(i + 1, rest - ZAHLEN[i]) or geht_falsch(i + 1, rest)
    gemerkt[rest] = ergebnis
    return ergebnis

print(geht_falsch(0, 7))
print(gemerkt)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
False
{1: False, 3: False, 4: False, 6: False, 5: False, 7: False}
```

Ausgegeben wird `False`, obwohl `3 + 2 + 2 = 7` ist! Im Wörterbuch steht `1: False`. Das stimmt für `i = 3` (nur noch die letzte 2 ist
übrig, `rest = 1` geht nicht), **aber nicht für `i = 0`**: Dort kann man die Zahl 1 benutzen. Der
Schlüssel `rest` allein reicht nicht, das Ergebnis hängt auch von `i` ab. Die Regel für den Schlüssel: **Er muss alles
enthalten, wovon das Ergebnis des Teilproblems abhängt**, nicht mehr (sonst merkt man zu wenig) und nicht weniger (sonst
stimmt das Ergebnis nicht). Hier gehören `i` **und** `rest` hinein.

</details>

#### Aufgabe 6.2

Korrigiere den Schlüssel in `geht_richtig`, sodass die Ausgabe stimmt.

**Erwartete Ausgabe:**
```
True
```

In [ ]:
ZAHLEN = (1, 3, 2, 2)
gemerkt_fehler = {}

def geht_richtig(i, rest):
    if rest == 0:
        return True
    if rest < 0 or i == len(ZAHLEN):
        return False
    if rest in gemerkt_fehler:
        return gemerkt_fehler[rest]
    ergebnis = geht_richtig(i + 1, rest - ZAHLEN[i]) or geht_richtig(i + 1, rest)
    gemerkt_fehler[rest] = ergebnis
    return ergebnis

print(geht_richtig(0, 7))

<details>
<summary>Tipp anzeigen</summary>

Nimm das Tupel `(i, rest)` als Schlüssel, und zwar an beiden Stellen (beim Nachsehen und beim Eintragen).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ZAHLEN = (1, 3, 2, 2)
gemerkt_fehler = {}

def geht_richtig(i, rest):
    if rest == 0:
        return True
    if rest < 0 or i == len(ZAHLEN):
        return False
    schluessel = (i, rest)               # beides gehört zum Teilproblem
    if schluessel in gemerkt_fehler:
        return gemerkt_fehler[schluessel]
    ergebnis = geht_richtig(i + 1, rest - ZAHLEN[i]) or geht_richtig(i + 1, rest)
    gemerkt_fehler[schluessel] = ergebnis
    return ergebnis

print(geht_richtig(0, 7))
```

Beide Stellen müssen denselben Schlüssel verwenden, sonst findet die Abfrage nie etwas oder liefert falsche Werte. Am
besten bildest du den Schlüssel einmal in einer Variablen `schluessel`.

</details>

## Wettbewerbs-Ecke 1: Schwimmbad

*Angelehnt an Aufgabe 2 „Schwimmbad“ der 1. Runde des 36. Bundeswettbewerbs Informatik (2017/18).*

Eine Gruppe will ins Schwimmbad und möchte so wenig wie möglich zahlen. Wir vereinfachen die Aufgabe: Es gibt **keine
Gutscheine** und keine Kinder unter vier Jahren. Jede Person ist **Jugendlicher** (`j`, bis 16 Jahre) oder **Erwachsener** (`e`,
über 16). Es gelten diese Karten (Preise in **Cent**, damit nicht mit Kommazahlen gerechnet werden muss):

| Karte | Wochenende | Wochentag |
|---|---|---|
| Einzelkarte Jugendlicher | 250 | 200 |
| Einzelkarte Erwachsener | 350 | 280 |
| Familienkarte (2 Erwachsene + 2 Jugendliche **oder** 1 Erwachsener + 3 Jugendliche; es darf auch weniger sein, aber mindestens ein Erwachsener) | 800 | 800 |
| Tageskarte für 6 Personen beliebigen Alters | gibt es nicht | 1100 |

Das Teilproblem ist: Was kostet es **mindestens**, `j` Jugendliche und `e` Erwachsene unterzubringen, an einem Wochenende oder an
einem Wochentag? Ein Zustand ist also das Tupel `(j, e, wochenende)`. Probiere für die **erste** Karte alle Möglichkeiten und
löse den Rest mit demselben Verfahren. Wir merken uns zu jeder Antwort auch die gekauften Karten (**Tupel** von Namen).

Die Funktion `beste_karten(j, e, wochenende)` gibt das Paar `(preis, karten)` zurück. Die Datei `daten/schwimmbad.txt` hat je
Zeile `Name;wochenende;jugendliche;erwachsene` (`wochenende` ist 1 oder 0).

#### Aufgabe E.1

Schreibe `beste_karten(j, e, wochenende)` mit dem Wörterbuch `gemerkt_bad` und dem Tupel `(j, e, wochenende)` als Schlüssel. Die
Hilfsfunktion `mit_karte(rest, preis, name)` fügt zu einem Ergebnis `(preis, karten)` eine Karte hinzu. Die Familienkarte
darf größer sein als die Restgruppe (dann zieht man mit `max(0, ...)` nicht mehr ab, als da sind).

**Erwartete Ausgabe:**
```
Antonia und Familie (Wochenende, 4 J, 1 E): 1050 Cent = 10,50 €
   1 x Jugendkarte, 1 x Familienkarte
Abiklasse am Wochentag (Wochentag, 0 J, 27 E): 5240 Cent = 52,40 €
   3 x Erwachsenenkarte, 4 x Tageskarte
Abiklasse am Wochenende (Wochenende, 0 J, 27 E): 9450 Cent = 94,50 €
   27 x Erwachsenenkarte
Familie Stutzenberg am Wochenende (Wochenende, 4 J, 5 E): 1950 Cent = 19,50 €
   1 x Erwachsenenkarte, 2 x Familienkarte
Familie Stutzenberg am Wochentag (Wochentag, 4 J, 5 E): 1700 Cent = 17,00 €
   3 x Jugendkarte, 1 x Tageskarte
Zwei Kegelvereine (Wochenende, 0 J, 32 E): 11200 Cent = 112,00 €
   32 x Erwachsenenkarte
Vier am Wochenende (Wochenende, 3 J, 1 E): 800 Cent = 8,00 €
   1 x Familienkarte
Klassenfahrt (Wochentag, 15 J, 12 E): 5000 Cent = 50,00 €
   3 x Jugendkarte, 4 x Tageskarte
278 Teilprobleme gemerkt
```

In [ ]:
FAMILIENKARTE = 800
TAGESKARTE = 1100
SORTEN = ['Jugendkarte', 'Erwachsenenkarte', 'Familienkarte', 'Tageskarte']

def einzelpreise(wochenende):
    if wochenende:
        return 250, 350                  # Jugendlicher, Erwachsener
    return 200, 280

def euro(cent):
    return str(cent // 100) + ',' + str(cent % 100).zfill(2) + ' €'

def mit_karte(rest, preis, name):
    """Ergebnis für den Rest, ergänzt um eine Karte."""
    rest_preis, rest_karten = rest
    return (rest_preis + preis, (name,) + rest_karten)

def lies_gruppen(dateiname):
    gruppen = []
    for zeile in open(dateiname, encoding='utf-8').read().strip().split('\n'):
        name, wochenende, j, e = zeile.split(';')
        gruppen.append((name, wochenende == '1', int(j), int(e)))
    return gruppen

gemerkt_bad = {}

def beste_karten(j, e, wochenende):
    return (0, ())                       # ersetze diese Zeile

for name, wochenende, j, e in lies_gruppen('daten/schwimmbad.txt'):
    preis, karten = beste_karten(j, e, wochenende)
    tag = 'Wochenende' if wochenende else 'Wochentag'
    print(name + ' (' + tag + ', ' + str(j) + ' J, ' + str(e) + ' E):', preis, 'Cent =', euro(preis))
    teile = []
    for sorte in SORTEN:
        if karten.count(sorte) > 0:
            teile.append(str(karten.count(sorte)) + ' x ' + sorte)
    print('  ', ', '.join(teile))
print(len(gemerkt_bad), 'Teilprobleme gemerkt')

<details>
<summary>Tipp anzeigen</summary>

Sammle alle Möglichkeiten für die erste Karte in einer Liste `moeglichkeiten` von Paaren `(preis, karten)`: für jede Karte
`rest = beste_karten(...)` mit der kleineren Gruppe und `mit_karte(rest, preis, name)`. Die Familienkarte gilt für `(erw, jug)`
= (2, 2) und (1, 3), die Tageskarte nur wochentags für `min(6, j + e)` Personen. Am Ende die Möglichkeit mit dem
kleinsten Preis nehmen, merken und zurückgeben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
FAMILIENKARTE = 800
TAGESKARTE = 1100
SORTEN = ['Jugendkarte', 'Erwachsenenkarte', 'Familienkarte', 'Tageskarte']

def einzelpreise(wochenende):
    if wochenende:
        return 250, 350                  # Jugendlicher, Erwachsener
    return 200, 280

def euro(cent):
    return str(cent // 100) + ',' + str(cent % 100).zfill(2) + ' €'

def mit_karte(rest, preis, name):
    """Ergebnis für den Rest, ergänzt um eine Karte."""
    rest_preis, rest_karten = rest
    return (rest_preis + preis, (name,) + rest_karten)

def lies_gruppen(dateiname):
    gruppen = []
    for zeile in open(dateiname, encoding='utf-8').read().strip().split('\n'):
        name, wochenende, j, e = zeile.split(';')
        gruppen.append((name, wochenende == '1', int(j), int(e)))
    return gruppen

gemerkt_bad = {}                         # (j, e, wochenende) -> (preis, karten)

def beste_karten(j, e, wochenende):
    if j == 0 and e == 0:
        return (0, ())                   # niemand mehr da: nichts zu bezahlen
    schluessel = (j, e, wochenende)
    if schluessel in gemerkt_bad:
        return gemerkt_bad[schluessel]
    preis_j, preis_e = einzelpreise(wochenende)
    moeglichkeiten = []                  # lauter Paare (preis, karten)
    if j > 0:                            # erste Karte: Einzelkarte Jugendlicher
        rest = beste_karten(j - 1, e, wochenende)
        moeglichkeiten.append(mit_karte(rest, preis_j, 'Jugendkarte'))
    if e > 0:                            # erste Karte: Einzelkarte Erwachsener
        rest = beste_karten(j, e - 1, wochenende)
        moeglichkeiten.append(mit_karte(rest, preis_e, 'Erwachsenenkarte'))
    if e > 0:                            # erste Karte: Familienkarte (2 E + 2 J oder 1 E + 3 J)
        for erw, jug in [(2, 2), (1, 3)]:
            rest = beste_karten(max(0, j - jug), max(0, e - erw), wochenende)
            moeglichkeiten.append(mit_karte(rest, FAMILIENKARTE, 'Familienkarte'))
    if not wochenende:                   # erste Karte: Tageskarte für 6 Personen (oder alle, wenn weniger)
        personen = min(6, j + e)
        for erw in range(0, min(e, personen) + 1):
            jug = personen - erw         # so viele Jugendliche passen noch dazu
            if jug <= j:
                rest = beste_karten(j - jug, e - erw, wochenende)
                moeglichkeiten.append(mit_karte(rest, TAGESKARTE, 'Tageskarte'))
    beste = moeglichkeiten[0]
    for moeglichkeit in moeglichkeiten:
        if moeglichkeit[0] < beste[0]:
            beste = moeglichkeit
    gemerkt_bad[schluessel] = beste
    return beste

for name, wochenende, j, e in lies_gruppen('daten/schwimmbad.txt'):
    preis, karten = beste_karten(j, e, wochenende)
    tag = 'Wochenende' if wochenende else 'Wochentag'
    print(name + ' (' + tag + ', ' + str(j) + ' J, ' + str(e) + ' E):', preis, 'Cent =', euro(preis))
    teile = []
    for sorte in SORTEN:
        if karten.count(sorte) > 0:
            teile.append(str(karten.count(sorte)) + ' x ' + sorte)
    print('  ', ', '.join(teile))
print(len(gemerkt_bad), 'Teilprobleme gemerkt')
```

Die Preise stimmen mit den Lösungen der echten Aufgabe überein (Lösungsnotebook Schwimmbad, dort noch mit Gutscheinen und
kleinen Kindern): Die Abiklasse zahlt wochentags 5240 Cent (vier Tageskarten für 24 Personen und drei Einzelkarten), am
Wochenende 9450. Die Großfamilie Stutzenberg zahlt am Wochenende 1950 (eine Einzelkarte und zwei Familienkarten) und
wochentags 1700 (eine Tageskarte und drei Jugendkarten). Bei „Vier am Wochenende“ ist eine Familienkarte (1 Erwachsener und 3
Jugendliche) mit 800 billiger als die Einzelkarten (1000).

</details>

Ein Wort zur **Ehrlichkeit**: `min` über alle Möglichkeiten für die erste Karte ist nur richtig, weil die Reihenfolge der Karten egal
ist und der **Rest** wieder ein Teilproblem derselben Art ist. Das nennt man das **Optimalitätsprinzip**: Ist die ganze Lösung
optimal, dann ist auch ihr Rest optimal für die Restgruppe (sonst könnte man den Rest verbessern und damit die ganze Lösung).

#### Frage E.2

1. Wie viele verschiedene Teilprobleme `(j, e, wochenende)` konnte die Funktion höchstens erzeugen, wenn alle Gruppen dieser Datei
   (größte Zahl Jugendlicher 15, größte Zahl Erwachsener 32) nacheinander berechnet werden? Wie viele hat sie tatsächlich gemerkt
   (siehe letzte Zeile der Ausgabe von E.1)?
2. Welche Arbeit steckt in einem Teilproblem, und was sagt das über die Laufzeit?
3. Warum braucht man `wochenende` im Schlüssel?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Höchstens (15 + 1) · (32 + 1) · 2 = **1056**: Jugendliche 0 bis 15, Erwachsene 0 bis 32, zwei Tagestypen. Tatsächlich gemerkt wurden
   **278**, weil nur Teilprobleme vorkommen, die von den Gruppen aus erreichbar sind. Das Wörterbuch bleibt
   zwischen den Gruppen erhalten, spätere Gruppen profitieren von früheren.
2. In jedem Teilproblem werden höchstens zwei Einzelkarten, zwei Familienkarten und sieben Tageskarten ausprobiert, also eine
   kleine **Konstante**. Die Laufzeit liegt bei **O(j · e)**. Ohne Merken würde sich die Zahl der Aufrufe mit jeder Person
   mehr vervielfachen.
3. Wochentags und am Wochenende kosten dieselben Gruppen verschieden viel (Einzelpreise, Tageskarte). Ließe man `wochenende` im
   Schlüssel weg, würde die zweite Rechnung die gemerkten Ergebnisse der ersten falsch verwenden. Das ist derselbe Fehler wie in
   Aufgabe 6.1. Alternativ leert man das Wörterbuch, wenn der Tagestyp wechselt.

</details>

**Zum Weiterdenken.** Bei der Klassenfahrt (15 Jugendliche und 12 Erwachsene an einem Wochentag) kostet es 5000 Cent. In der echten Aufgabe kommen noch Gutscheine
dazu (nicht für jede Person, sondern 10 % für die Gruppe oder ein freier Eintritt), und Kinder unter vier Jahren. Siehe das
[Lösungsnotebook zum Schwimmbad](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/36_Schwimmbad/schwimmbad_loesung.ipynb): Dort werden die Gutscheine **von außen** durchprobiert und
für jede Verteilung `beste_karten` aufgerufen. Das Wörterbuch macht diese vielen Aufrufe erst bezahlbar.

## Wettbewerbs-Ecke 2: Nummernmerker (für Schnelle)

*Angelehnt an Aufgabe 2 „Nummernmerker“ der 1. Runde des 38. Bundeswettbewerbs Informatik (2019/20).*

Sarah merkt sich lange Nummern in Blöcken. Jeder Block hat **2 bis 4 Ziffern**, und die Zahl der Blöcke, die mit einer **Null
beginnen**, soll möglichst klein sein. Aus `01365400606` wird zum Beispiel `0136 5400 606`: Ein Block beginnt mit einer Null.

Das Teilproblem: Wie viele Blöcke mit führender Null braucht man **mindestens**, wenn man nur noch die Ziffern **ab Position `ab`**
aufteilt? Der erste Block hat Länge 2, 3 oder 4 und kostet 1, wenn an Position `ab` eine `0` steht, sonst 0. Dazu kommen die
Kosten für den Rest. Ist alles aufgeteilt (`ab == len(nummer)`), kostet es 0. Bleibt eine einzelne Ziffer übrig, geht es nicht:
Das zeigt `UNMOEGLICH = 1000` an, eine so große Zahl, dass diese Möglichkeit nie gewinnt.

Der Schlüssel ist das Tupel `(nummer, ab)`, damit man das Wörterbuch nicht für jede Nummer leeren muss (die Nummer ist ein Text, also
unveränderlich). `wahl_nr` merkt, wie lang der erste Block der besten Möglichkeit ist, damit man die Blöcke am Ende wieder
zusammensetzen kann.

#### Aufgabe E.3

Schreibe `beste_nummer(nummer, ab)` mit den Wörterbüchern `gemerkt_nr` und `wahl_nr` (Schlüssel `(nummer, ab)`) und
`bloecke(nummer)`, das die Blöcke einer besten Aufteilung als Liste von Texten zurückgibt (Zurückverfolgen mit `wahl_nr`).
`nummer[ab:ab + laenge]` ist der Block der Länge `laenge` ab der Position `ab`.

**Erwartete Ausgabe:**
```
005480000005179734 -> 00 54 800 0000 51 79 734 | mit führender Null: 2
03495929533790154412660 -> 03 49 59 29 53 37 90 15 44 12 660 | mit führender Null: 1
5319974879022725607620179 -> 53 19 97 48 790 22 72 560 76 20 179 | mit führender Null: 0
9088761051699482789038331267 -> 90 88 76 10 51 69 94 82 78 90 38 33 12 67 | mit führender Null: 0
011000000011000100111111101011 -> 01 10 000 0001 1000 100 11 11 11 10 10 11 | mit führender Null: 3
```

In [ ]:
UNMOEGLICH = 1000

def lies_nummern(dateiname):
    return open(dateiname).read().split()

gemerkt_nr = {}
wahl_nr = {}

def beste_nummer(nummer, ab):
    return 0                             # ersetze diese Zeile

def bloecke(nummer):
    liste = []
    # dein Code
    return liste

for nummer in lies_nummern('daten/nummern.txt'):
    kosten = beste_nummer(nummer, 0)
    print(nummer, '->', ' '.join(bloecke(nummer)), '| mit führender Null:', kosten)

<details>
<summary>Tipp anzeigen</summary>

Wie bei den Münzen: Für jede Länge 2, 3, 4, die noch in die Nummer passt, die Kosten des Rests `beste_nummer(nummer, ab + laenge)`
holen, 1 addieren, wenn `nummer[ab] == '0'` ist, und bei einer Verbesserung `beste` und `wahl_nr[(nummer, ab)]` setzen. In
`bloecke`: mit `ab = 0` beginnen, die Länge aus `wahl_nr` holen, den Block anhängen, `ab` weiterschieben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
UNMOEGLICH = 1000

def lies_nummern(dateiname):
    return open(dateiname).read().split()

gemerkt_nr = {}                          # (nummer, ab) -> wenigste Blöcke mit führender Null
wahl_nr = {}                             # (nummer, ab) -> Länge des ersten Blocks der besten Möglichkeit

def beste_nummer(nummer, ab):
    if ab == len(nummer):
        return 0
    schluessel = (nummer, ab)
    if schluessel in gemerkt_nr:
        return gemerkt_nr[schluessel]
    beste = UNMOEGLICH
    for laenge in [2, 3, 4]:
        if ab + laenge <= len(nummer):
            kosten = beste_nummer(nummer, ab + laenge)
            if nummer[ab] == '0':        # der erste Block beginnt mit einer Null
                kosten = kosten + 1
            if kosten < beste:
                beste = kosten
                wahl_nr[schluessel] = laenge
    gemerkt_nr[schluessel] = beste
    return beste

def bloecke(nummer):
    liste = []
    ab = 0
    beste_nummer(nummer, 0)              # füllt wahl_nr
    while ab < len(nummer):
        laenge = wahl_nr[(nummer, ab)]
        liste.append(nummer[ab:ab + laenge])
        ab = ab + laenge
    return liste

for nummer in lies_nummern('daten/nummern.txt'):
    kosten = beste_nummer(nummer, 0)
    print(nummer, '->', ' '.join(bloecke(nummer)), '| mit führender Null:', kosten)
```

Beginnt eine Nummer mit einer `0`, hat schon der erste Block eine führende Null, daran lässt sich nichts ändern. Die
Aufteilungen sind nicht eindeutig: Bei mehreren gleich guten gewinnt die zuerst probierte (kürzester erster Block). Wichtig ist
die Zahl der Blöcke mit führender Null, und die ist bei jeder Nummer minimal.

</details>

#### Frage E.4

1. Wie viele Aufrufe macht `beste_nummer` höchstens für eine Nummer mit `n` Ziffern (mit Merken)? Wie groß ist die Laufzeit?
2. Ohne Merken sind es für die 28-stellige Nummer 50 897 Aufrufe. Warum wachsen sie ohne Merken so stark?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Teilprobleme sind die Positionen `ab = 0, 1, ..., n`, also **n + 1** Stück (die Nummer gehört zum Schlüssel, ändert
   sich aber innerhalb einer Berechnung nicht). Jedes macht höchstens drei Aufrufe für die Blocklängen 2, 3, 4. Die Laufzeit ist
   **O(n)**, für 30 Ziffern höchstens etwa 100 Aufrufe.
2. Zu jedem Anfang kann man auf mehreren Wegen kommen (zum Beispiel 2 + 4 und 3 + 3 und 4 + 2 Ziffern für die ersten sechs), und
   ohne Merken wird der Rest jedes Mal neu berechnet. Die Zahl der Aufrufe wächst deshalb **exponentiell** mit `n`, mit
   Merken nur linear.

</details>

## Das hast du gelernt

```python
# Muster Memoisierung: Schlüssel = Tupel aller Angaben, von denen das Ergebnis abhängt
gemerkt = {}

def loese(a, b):
    if a == 0 and b == 0:                # einfacher Fall zuerst
        return 0
    schluessel = (a, b)
    if schluessel in gemerkt:            # schon berechnet?
        return gemerkt[schluessel]
    ergebnis = ...                       # Rest mit loese(...) lösen; summieren, min, max oder or
    gemerkt[schluessel] = ergebnis       # eintragen, dann zurückgeben
    return ergebnis

# Dasselbe automatisch, die Argumente müssen unveränderlich sein (keine Listen)
from functools import lru_cache

@lru_cache(maxsize=None)
def loese(a, b):
    ...                                  # ohne Wörterbuch; loese.cache_clear() leert den Speicher
```

- **Memoisierung**: Vor dem Rechnen im Wörterbuch nachsehen, nach dem Rechnen eintragen. Jedes Teilproblem wird nur einmal berechnet.
- **Schlüssel**: ein Tupel, das **alles** enthält, wovon das Ergebnis abhängt. Zu wenig ergibt falsche Ergebnisse, Listen gehen nicht.
- **Zählen** (`+`), **das Beste** (`min`, `max`) und **geht es?** (`or`) lassen sich auf dieselbe Weise merken. Auch ein Ergebnis wie `None`
  oder `False` muss man merken.
- **Laufzeit**: Zahl der verschiedenen Teilprobleme × Arbeit je Teilproblem. Ohne Merken wächst die Zahl der Aufrufe oft
  exponentiell, mit Merken oft nur wie das Produkt der Größen der Schlüsselangaben.
- **Lösung zurückverfolgen**: je Teilproblem die **beste erste Entscheidung** merken (oder nachträglich neu finden) und der Reihe nach abarbeiten.
- **`@lru_cache(maxsize=None)`** ersetzt Wörterbuch, Abfrage und Eintragen. Argumente unveränderlich, globale Variablen sind eine Falle,
  `cache_clear()` leert den Speicher. Die Rekursionstiefe bleibt begrenzt (etwa 1000).

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Schwimmbad](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/36_Schwimmbad/schwimmbad_loesung.ipynb) (36. BwInf): billigste Karten für eine Gruppe, mit Gutscheinen
- [Nummernmerker](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Nummernmerker/nummernmerker_loesung.ipynb) (38.): Nummern in Blöcke teilen
- [Flaschenzug](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/34_Flaschenzug/flaschenzug_loesung.ipynb) (34.): Zwischenergebnisse merken
- [Wandertag](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/43_Wandertag/wandertag_loesung.ipynb) (43.): Zwischenergebnisse merken

Im nächsten Notebook (11) drehst du die Rekursion um: Statt von oben nach unten und mit Wörterbuch füllst du eine **Tabelle von
klein nach groß**. Das umgeht die Grenze der Rekursionstiefe. Außerdem lernst du **Präfixsummen** kennen, mit denen man Summen über
beliebige Abschnitte einer Liste mit einer einzigen Subtraktion berechnet.